# ColumnTransformer preprocessing practice

Original learning notebook with assisted reproducibility corrections. Manual train and test feature order now matches; the dataset path is relative, the split is fixed and stratified, and one-hot encoding handles unseen categories. Outputs are cleared because the fixed split changes the data. Run all cells again and explain the corrections independently. Next: add a model inside a Pipeline.


In [ ]:
import numpy as np
import pandas as pd


In [ ]:
from pathlib import Path

# Run from this folder or the repository root.
data_path = Path("Datasets/covid_toy.csv")
if not data_path.exists():
    data_path = Path("Column Transformer") / data_path
df = pd.read_csv(data_path)

In [ ]:
df.head()

In [ ]:
df['cough'].value_counts()

In [ ]:
df.info()

### Without Column Transformer

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    df.drop(columns=['has_covid']), df['has_covid'],
    test_size=0.3, random_state=42, stratify=df['has_covid']
)


In [ ]:
#imputing fever
from sklearn.impute import SimpleImputer
si=SimpleImputer()

x_train_fever=si.fit_transform(x_train[['fever']])
x_test_fever=si.transform(x_test[['fever']])


In [ ]:
x_train_fever.shape,x_test_fever.shape

In [ ]:
# Ordinal Encoding
from sklearn.preprocessing import OrdinalEncoder
oe=OrdinalEncoder(categories=[['Mild','Strong']])
x_train_cough=oe.fit_transform(x_train[['cough']])
x_test_cough=oe.transform(x_test[['cough']])

In [ ]:
x_train_cough.shape,x_test_cough.shape

In [ ]:
#OneHot Encoding
from sklearn.preprocessing import OneHotEncoder
ohe=OneHotEncoder(drop='first', handle_unknown='ignore')
x_train_gender_city=ohe.fit_transform(x_train[['gender','city']]).toarray()
x_test_gender_city=ohe.transform(x_test[['gender','city']]).toarray()

In [ ]:
x_train_gender_city.shape, x_test_gender_city.shape

In [ ]:
# Extracting Age
x_train_age = x_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
x_test_age = x_test.drop(columns=['gender','fever','cough','city']).values

x_train_age.shape,x_test_age.shape

In [ ]:
final_df=np.concatenate((x_train_age,x_train_cough,x_train_gender_city,x_train_fever),axis=1)
final_df.shape

In [ ]:
x_test_transformed = np.concatenate((x_test_age,x_test_cough,x_test_gender_city,x_test_fever),axis=1)
x_test_transformed.shape

### With column transformer

In [ ]:
from sklearn.compose import ColumnTransformer
transformer=ColumnTransformer(transformers=[
    ('tf1',SimpleImputer(),['fever']),
    ('tf2',OneHotEncoder(drop='first', handle_unknown='ignore'),['gender','city']),
    ('tf3',OrdinalEncoder(categories=[['Mild','Strong']]),['cough'])],remainder='passthrough')  # by default remainder=drop

x_train_new=transformer.fit_transform(x_train)
x_test_new=transformer.transform(x_test)


In [ ]:
x_train_new.shape,x_test_new.shape